In [1]:
## imports

import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [2]:
## Load ANN Trained model, Scalar pickle file, and OHE pickle file

model = load_model('model.keras')

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('ohe_geo.pkl', 'rb') as file:
    ohe_geo = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [3]:
## Example input data

input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 30,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [4]:
## OHE geography variable

geo_encoded = ohe_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns = ohe_geo.get_feature_names_out(['Geography']))
geo_encoded_df

c:\Users\shree\Documents\Study\GenAI with Langchain & Huggingface\Projects\annclassification\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [5]:
## Convert to dataframe

input_data = pd.DataFrame([input_data])
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,30,3,60000,2,1,1,50000


In [6]:
## Encode Gender Variable

input_data['Gender'] = label_encoder_gender.transform(input_data['Gender'])
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,30,3,60000,2,1,1,50000


In [7]:
## Concatenate dfs

input_df = pd.concat(
    [input_data.drop('Geography', axis = 1), geo_encoded_df],
    axis = 1
)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,30,3,60000,2,1,1,50000,1.0,0.0,0.0


In [8]:
## Scaling input data

input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755, -0.84593077, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [9]:
## Predict Churn

prediction = model.predict(input_scaled)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
[[0.01271149]]


In [10]:
prediction_prob = prediction[0][0]

if prediction_prob > 0.5:
    print(f'The prediction probability of {prediction_prob} is greater than 0.5. \nTherefore, the customer is likely to churn.')
else:
    print(f'The prediction probability of {prediction_prob} is less than 0.5. \nTherefore, the customer is not likely to churn.')

The prediction probability of 0.012711486779153347 is less than 0.5. 
Therefore, the customer is not likely to churn.
